# From Photo to BIM: Extracting Structured Data from Imagery

This notebook demonstrates a workflow for converting **unstructured photo data** into a **structured, geolocated dataset** linked to a BIM model.

The pattern applies to any domain where field photos need to be documented, geolocated, and brought into CAD/BIM platforms — asset management, construction documentation, infrastructure inspections, heritage surveys, and more.

### Workflow Overview

| Step | What happens |
|------|-------------|
| 1. GPS Extraction | Read latitude/longitude from the photo's EXIF metadata |
| 2. Coordinate Transform | Convert WGS84 (GPS) coordinates to a local projected grid |
| 3. Image Analysis | Send the photo to a vision-language model for description |
| 4. Data Assembly | Combine GPS, coordinates, and LLM response into a structured record |
| 5. Visualisation | Plot photo locations on an interactive map |
| 6. Export | Write the structured data to CSV for BIM ingestion |

### What you need

- A JPEG photo with GPS metadata (most phones and drones embed this automatically)
- A `GROQ_API_KEY` in your `.env` file (free tier available at [groq.com](https://groq.com))
- The Python packages listed in `requirements.txt`

---

## Setup

In [1]:
import os
import base64
from fractions import Fraction
from math import floor
from pathlib import Path

import pandas as pd
from PIL import Image, ExifTags
from pyproj import Transformer
from dotenv import load_dotenv
import folium

load_dotenv()

if not os.environ.get("GROQ_API_KEY"):
    raise SystemExit("GROQ_API_KEY not found. Create a .env file with your API key.")

---

## 1. GPS Extraction

Most cameras (including smartphones and drones) embed GPS coordinates in JPEG photos as **EXIF metadata**. This step reads those coordinates so we know *where* the photo was taken.

EXIF stores GPS in degrees-minutes-seconds (DMS) format, so we convert to decimal degrees.

In [2]:
GPSINFO_TAG = next(tag for tag, name in ExifTags.TAGS.items() if name == "GPSInfo")


def _to_float(x):
    return float(x) if isinstance(x, Fraction) else x


def _gps_to_decimal(value):
    """Convert a GPS value from DMS tuple or decimal to a float."""
    if isinstance(value, (Fraction, float, int)):
        return _to_float(value)
    if isinstance(value, (tuple, list)) and len(value) == 3:
        d, m, s = (_to_float(v) for v in value)
        return d + m / 60 + s / 3600
    raise TypeError(f"Unrecognised GPS value format: {type(value)}")


def extract_gps_from_image(image_path: str) -> tuple[float, float] | None:
    """Extract (latitude, longitude) from JPEG EXIF GPS tags.

    Returns None if the image has no GPS data.
    """
    img = Image.open(image_path)
    exif = img.getexif()
    gps_ifd = exif.get_ifd(GPSINFO_TAG)
    if not gps_ifd:
        return None

    gps = {ExifTags.GPSTAGS.get(k, k): v for k, v in gps_ifd.items()}
    if "GPSLatitude" not in gps or "GPSLongitude" not in gps:
        return None

    lat = _gps_to_decimal(gps["GPSLatitude"])
    lon = _gps_to_decimal(gps["GPSLongitude"])

    if gps.get("GPSLatitudeRef") == "S" and lat > 0:
        lat = -lat
    if gps.get("GPSLongitudeRef") == "W" and lon > 0:
        lon = -lon

    return float(lat), float(lon)

In [3]:
IMAGE_PATH = "MVIMG_20260212_194030.jpg"

coords = extract_gps_from_image(IMAGE_PATH)
if coords is None:
    raise SystemExit(f"No GPS data found in {IMAGE_PATH}")

latitude, longitude = coords
print(f"Latitude:  {latitude:.7f}")
print(f"Longitude: {longitude:.7f}")

Latitude:  -27.2186238
Longitude: 153.0967341


---

## 2. Coordinate Transformation

GPS coordinates are in **WGS84** (EPSG:4326) — a global system using latitude and longitude.

BIM and CAD platforms typically work in a **projected coordinate system** — a flat grid measured in metres, tied to a specific region. This step converts the GPS coordinates into that local grid.

Here we convert to **MGA Zone 56 (GDA94)**, which covers parts of eastern Australia. Replace the EPSG code with whatever projected system your project uses.

In [4]:
def lon_to_utm_zone(lon: float) -> int:
    """Calculate the UTM zone number from a longitude value."""
    return int(floor((lon + 180) / 6) + 1)


def epsg_for_mga94(zone: int) -> int:
    """EPSG code for MGA94 at a given UTM zone (e.g. zone 56 → 28356)."""
    return 28300 + zone


def wgs84_to_local_grid(lat: float, lon: float, target_epsg: int) -> tuple[float, float]:
    """Convert WGS84 lat/lon to a projected coordinate system.

    Returns (easting, northing) in metres.
    """
    transformer = Transformer.from_crs(4326, target_epsg, always_xy=True)
    easting, northing = transformer.transform(lon, lat)
    return easting, northing

In [5]:
# Choose the projected coordinate system for your project
zone = lon_to_utm_zone(longitude)
target_epsg = epsg_for_mga94(zone)  # 28356 for Zone 56

easting, northing = wgs84_to_local_grid(latitude, longitude, target_epsg)

print(f"UTM Zone:       {zone}")
print(f"EPSG code:      {target_epsg}")
print(f"Easting:        {easting:.3f} m")
print(f"Northing:       {northing:.3f} m")

UTM Zone:       56
EPSG code:      28356
Easting:        509578.865 m
Northing:       6989346.086 m


---

## 3. Image Analysis

We send the photo to a **vision-language model** to generate a structured description of what's visible.

Two important steps happen before the image reaches the model:

1. **Resize** — API providers have payload size limits (Groq's is 4 MB). We scale the image down to stay within that.
2. **Strip EXIF** — The resized image is re-saved without metadata, so GPS coordinates and camera details are **never sent** to the external API.

In [6]:
from groq import Groq

MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"


def prepare_image_for_api(src_path: str, dst_path: str,
                          scale: float = 0.5, quality: int = 80) -> str:
    """Resize and EXIF-strip an image for API submission.

    scale=0.5 halves both dimensions (~75% file size reduction).
    The re-save naturally strips all EXIF metadata.
    """
    img = Image.open(src_path)
    if img.mode != "RGB":
        img = img.convert("RGB")

    w, h = img.size
    img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    img.save(dst_path, "JPEG", quality=quality, optimize=True)

    size_bytes = os.path.getsize(dst_path)
    if size_bytes > 4_000_000:
        print(f"Warning: resized image is {size_bytes:,} bytes — may exceed API limit.")

    return dst_path


def image_to_base64_data_url(path: str) -> str:
    """Read a JPEG file and encode it as a base64 data URL."""
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

In [7]:
def analyze_image(image_path: str, prompt: str, api_key: str) -> str:
    """Send an image to a vision-language model and return its text response.

    The image is resized and EXIF-stripped before sending.
    """
    # Resize into a temp file (EXIF stripped)
    src = Path(image_path)
    dst = src.with_name(src.stem + "_small" + src.suffix)
    prepare_image_for_api(str(src), str(dst))

    data_url = image_to_base64_data_url(str(dst))

    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
        max_completion_tokens=300,
        temperature=0.2,
    )

    return completion.choices[0].message.content

In [8]:
# Prompt for a general scene description (good for validation / testing)
SCENE_DESCRIPTION_PROMPT = "Describe this image in 3-5 factual sentences."

llm_response = analyze_image(IMAGE_PATH, SCENE_DESCRIPTION_PROMPT, os.environ["GROQ_API_KEY"])
print(llm_response)

The image shows a bowl of food on a table with a sports field in the background. 

The bowl contains fried fish, herbs, and a lemon wedge. A glass of champagne is to the left of the bowl. The table is made of light-colored wood and has a white napkin with a logo on it. In the background, there is a sports field with people playing a game, and spectators watching from the stands. The atmosphere suggests that the photo was taken at a sports event, possibly a rugby or soccer match, given the presence of a large crowd and the field.


---

## 4. Structured Data Assembly

Now we combine everything — the image name, GPS coordinates, projected coordinates, and the LLM's analysis — into a single structured record.

This is the output that gets consumed by BIM platforms, GIS tools, or inspection databases.

In [9]:
def build_inspection_record(image_path: str,
                            latitude: float,
                            longitude: float,
                            zone: int,
                            easting: float,
                            northing: float,
                            llm_response: str) -> pd.DataFrame:
    """Create a one-row DataFrame with all collected data."""
    return pd.DataFrame([{
        "image_name": Path(image_path).name,
        "latitude": latitude,
        "longitude": longitude,
        "zone": zone,
        "easting_m": round(easting, 3),
        "northing_m": round(northing, 3),
        "llm_response": llm_response,
    }])

In [10]:
record = build_inspection_record(
    image_path=IMAGE_PATH,
    latitude=latitude,
    longitude=longitude,
    zone=zone,
    easting=easting,
    northing=northing,
    llm_response=llm_response,
)
record

,image_name,latitude,longitude,zone,easting_m,northing_m,llm_response
0,MVIMG_20260212_194030.jpg,-27.218624,153.096734,56,509578.865,6989346.086,The image shows a bowl of food on a table with...


---

## 5. Map Visualisation

As a validation step, we plot the photo's GPS location on an interactive map. This lets you quickly confirm that the extracted coordinates look correct before feeding them into your BIM model.

In [11]:
def create_location_map(df: pd.DataFrame,
                        output_path: str = "inspection_map.html") -> folium.Map:
    """Create an interactive map with markers at each photo's location.

    Clicking a marker shows the image name and LLM description.
    The map is saved as an HTML file you can open in any browser.
    """
    center_lat = df["latitude"].mean()
    center_lon = df["longitude"].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

    for _, row in df.iterrows():
        popup_html = f"<b>{row['image_name']}</b><br>{row['llm_response']}"
        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(m)

    m.save(output_path)
    print(f"Map saved to {output_path}")
    return m

In [12]:
inspection_map = create_location_map(record)
inspection_map

Map saved to inspection_map.html


---

## 6. Export

The final step is writing the structured data to CSV. This file can be imported into Revit (via Dynamo), Civil 3D, GIS tools, or any system that accepts tabular data.

### Dynamo Integration

A sample Dynamo graph (`Image_to_Revit_demo.dyn`) is included. It reads this CSV, converts the projected coordinates to Revit's internal coordinate system, and places a marker family at the correct location with the LLM description attached as a parameter.

In [13]:
OUTPUT_CSV = "inspection_records.csv"
record.to_csv(OUTPUT_CSV, index=False)
print(f"Exported to {OUTPUT_CSV}")

Exported to inspection_records.csv


---

## Appendix: Domain-Specific Analysis Prompt

The scene description prompt above is good for general use. If you're doing infrastructure inspections, you can swap in a more structured prompt like this one:

```python
ASSET_INSPECTION_PROMPT = \"\"\"
You are analysing a photo of infrastructure or a built asset.

Identify visible conditions in these categories:
- Surface defects (cracks, spalls, corrosion)
- Missing or damaged elements
- Structural concerns
- Other notable observations

Rules:
- Bullet points only
- One sentence per observation
- Be specific but concise
- Do NOT estimate dimensions without scale reference
- Do NOT speculate on causes
\"\"\"

llm_response = analyze_image(IMAGE_PATH, ASSET_INSPECTION_PROMPT, os.environ["GROQ_API_KEY"])
```

The key principle: **constrain the prompt tightly** to reduce hallucination and keep outputs consistent enough for downstream processing.